In [ ]:
import requests, hashlib, shutil, os, json, socket, multiprocessing, configparser, sys
from scipy.io import savemat, loadmat
from scipy import signal
from tqdm import tqdm
import numpy as np
import pandas as pd

settings_parser = configparser.ConfigParser()
settings_parser.read("localsettings.ini")
MAIN_DATA_FOLDER = settings_parser.get("global", "data_path")
if not os.path.isdir(MAIN_DATA_FOLDER):
    os.mkdir(MAIN_DATA_FOLDER)
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
if not os.path.isdir(RESULTS_FOLDER):
    os.mkdir(RESULTS_FOLDER)
MIENC_PATH = settings_parser.get("global", "mienc_path")
sys.path.append(MIENC_PATH)
from mienc.support import surrogate

config_ini = os.path.join(MAIN_DATA_FOLDER, "config.ini")
block_size = 1024
config = configparser.ConfigParser()
seconds_for_chunk = 124

bands = {
    i: b
    for i, b in enumerate(
        [
            [1.0, 4.0],
            [4.0, 8.0],
            [8.0, 12.0],
            [12.0, 30.0],
            [30.0, 44.0],
        ],
        start=1,
    )
}
band_names = {
    i: b
    for i, b in enumerate(
        ["delta", "theta", "alpha", "beta", "gamma"],
        start=1,
    )
}


def fs_band(band):
    return int(bands[band][1] * 2 * 1.125 + 0.5)

In [ ]:
import tarfile, mne, tempfile, base64

In [ ]:
# https://fcon_1000.projects.nitrc.org/indi/retro/MPI_LEMON/downloads/download_EEG.html
# Babayan, A.; Erbey, M.; Kumral, D.; Reinelt, J.; Reiter, A.; Röbbig, J.; Schaare, H. L.; Ragert, M.; Anwander, A.; Bazin, P.-L. ; Horstmann, A.; Lampe, L.; Nikulin, V. V.; Okon-Singer, H.; Preusser, S.; Pampel, A.; Rohr, C. S.; Sacher, J.; Thöne-Otto, A. I. T.; Trapp, S.; Nierhaus, T.; Altmann, D.; Arélin, K.; Blöchl, M.; Bongartz, E.; Breig, P.; Cesnaite, E.; Chen , S.; Cozatl, R.; Czerwonatis, S.; Dambrauskaite, G.; Paerisch, M.; Enders , J.; Engelhardt , M.; Fischer , M. M.; Forschack, N.; Golchert, J.; Golz, L.; Guran, C. A.; Hedrich, S.; Hentschel, N.; Hoffmann , D. I.; Huntenburg, J. M.; Jost , R.; Kosatschek, A.; Kunzendorf, S.; Lammers, H.; Lauckner, M.; Mahjoory, K.; Kanaan, A. S.; Mendes, N.; Menger, R.; Morino, E.; Naethe, K.; Neubauer , J.; Noyan, H.; Oligschläger, S.; Panczyszyn-Trzewik, P.; Poehlchen, D.; Putzke, N.; Roski, S.; Schaller , M.-C.; Schieferbein, A.; Schlaak, B.; Schmidt, R.; Gorgolewski, K. J.; Schmidt, H. M.; Schrimpf, A.; Stasch, S.; Voss, M.; Wiedemann, A.; Margulies, D. S.; Gaebler, M.; Villringer, A.: A mind-brain-body dataset of MRI, EEG, cognition, emotion, and peripheral physiology in young and old adults. Scientific Data 6 (2019)
import tarfile, mne, tempfile, base64

missing_sub = [
    32335,
    32384,
    32404,
    32433,
    32437,
    32443,
    32445,
    32461,
    32488,
    32500,
    32519,
    32520,
    32527,
]
missing_aws = [
    32309,
    32365,
    32366,
    32374,
    32382,
    32410,
    32419,
    32426,
    32485,
    32486,
    32487,
    32489,
    32492,
]
bad_subjects = [
    32301,
    32306,
    32319,
    32323,
    32326,
    32329,
    32334,
    32336,
    32342,
    32343,
    32350,
    32354,
    32355,
    32357,
    32360,
    32362,
    32367,
    32368,
    32370,
    32372,
    32379,
    32380,
    32381,
    32383,
    32387,
    32396,
    32397,
    32399,
    32402,
    32417,
    32428,
    32434,
    32436,
    32439,
    32447,
    32449,
    32450,
    32466,
    32468,
    32469,
    32478,
    32493,
    32497,
    32498,
    32510,
    32512,
    32514,
    32521,
    32522,
    32523,
    32525,
    32526,
]
bad_electrodes = ["T7", "T8", "Cz", "F7", "CP6", "PO10", "Fp2"]
missing_ids = missing_sub + missing_aws + bad_subjects
first_id = 32301
last_id = 32528
tot_subjects = last_id - first_id - len(missing_ids) + 1

output_EEG = os.path.join(MAIN_DATA_FOLDER, "EEG")

tmp_dir = os.path.join(output_EEG, f"temporary_arrays")
if not os.path.isdir(tmp_dir):
    os.mkdir(tmp_dir)
    
mne.set_log_level(verbose="error")

In [ ]:
54*53/2*150/945750

In [ ]:
sid = 32304
qwerty = requests.get(f"https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/MPI-LEMON/Compressed_tar/EEG_MPILMBB_LEMON/EEG_Localizer_BIDS_ID/sub-{sid:06}.tar.gz")
import io
from scipy.io import loadmat
if qwerty.status_code==200:
    cont = io.BytesIO(qwerty.content)
    tar = tarfile.open(fileobj=cont, mode="r:gz")
    baz = tar.extractfile(f"sub-{sid:06}/sub-{sid:06}.mat").read()
    file = io.BytesIO(baz)
    mat = loadmat(file)
    print(mat.keys())
else:
    print(qwerty.reason)

In [ ]:
chan_pos = {}
for a in mat["Channel"][0,:]:
    chan_pos[a[0][0].split("_")[-1]] = a[-3][:,0]

In [ ]:
all_electrodes-chan_pos.keys()
chan_pos.keys()-all_electrodes

In [ ]:
all_positions = {}
with requests.Session() as session, tqdm(
    total=tot_subjects, desc="Subjects"
) as main_pbar:
    for sid in range(first_id, last_id + 1):
        if sid in missing_ids:
            continue
        if not os.path.isfile(os.path.join(output_EEG, f"sub-{sid:06}.tar.gz")):
            try:
                lemon_file_response = session.get(
                    f"https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/MPI-LEMON/Compressed_tar/EEG_MPILMBB_LEMON/EEG_Preprocessed_BIDS_ID/EEG_Preprocessed/sub-{sid:06}.tar.gz",
                    stream=True,
                )
                file_size = int(lemon_file_response.headers["Content-Length"])
                with tqdm(
                    desc=f"Subject {sid}",
                    unit="B",
                    unit_scale=True,
                    total=file_size,
                    leave=False,
                ) as progress_bar:
                    with open(
                        os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp"), "wb"
                    ) as fp:
                        for data in lemon_file_response.iter_content(block_size):
                            progress_bar.update(len(data))
                            fp.write(data)

                assert (
                    file_size
                    == os.stat(
                        os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp")
                    ).st_size
                )
                os.rename(
                    os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp"),
                    os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"),
                )
            except Exception as e:
                if os.path.isfile(os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp")):
                    os.unlink(os.path.join(output_EEG, f"sub-{sid:06}.tar.gz.tmp"))
                print(e)
        lemon_localiser_response = session.get(f"https://fcp-indi.s3.amazonaws.com/data/Projects/INDI/MPI-LEMON/Compressed_tar/EEG_MPILMBB_LEMON/EEG_Localizer_BIDS_ID/sub-{sid:06}.tar.gz")
        if lemon_localiser_response.status_code==200:
            content = io.BytesIO(lemon_localiser_response.content)
            tar = tarfile.open(fileobj=content, mode="r:gz")
            inflated = io.BytesIO(tar.extractfile(f"sub-{sid:06}/sub-{sid:06}.mat").read())
            positions = loadmat(inflated)["Channel"][0,:]
            for pos in positions:
                try:
                    all_positions[pos[0][0].split("_")[-1]].append(pos[-3].squeeze())
                except:
                    all_positions[pos[0][0].split("_")[-1]] = [pos[-3].squeeze()]
        main_pbar.update()

In [ ]:
average_positions = []
for key in all_positions:
    if key in bad_electrodes+['FP2', 'Reference', 'VEOG']:
        continue
    tmp = []
    for a in all_positions[key]:
        tmp.append(a.squeeze())
    tmp = np.stack(tmp)
    if key=="FP1":
        key="Fp1"
    average_positions.append([key,*tmp.mean(0).tolist()])
electrode_positions=pd.DataFrame(average_positions, columns=["Labels", "y", "x", "z"])
electrode_positions.to_csv(os.path.join(output_EEG, "electrode_positions.csv"))


In [ ]:
from scipy.spatial import Voronoi
import matplotlib.pyplot as plt
plane_distance=10
source_z = electrode_positions.z.min()
diameter = electrode_positions.z.max() - source_z
electrode_positions["XP"] = (
    electrode_positions.x
    / (electrode_positions.z - source_z + plane_distance)
    * (diameter + plane_distance)
)
electrode_positions["YP"] = (
    electrode_positions.y
    / (electrode_positions.z - source_z + plane_distance)
    * (diameter + plane_distance)
)

vor = Voronoi(#electrode_positions[["XP", "YP"]],
    np.concatenate(
        [
            electrode_positions[["XP", "YP"]],
            np.transpose(
                [
                    0.15 * np.sin(np.linspace(0, 2 * np.pi, 100)),
                    0.15 * np.cos(np.linspace(0, 2 * np.pi, 100)),# - 0.5,
                ]
            ),
        ]
    )
)
for i, v in enumerate(average_positions):
    region = vor.regions[vor.point_region[i]]
    vertices = np.array([vor.vertices[n] for n in region])
    plt.fill(vertices[:, 0], vertices[:, 1], color=(i/53, 0, 1-i/53))
plt.gca().set_aspect("equal")

In [ ]:
from s03a_localisation import plot_cap, electrode_positions

In [ ]:
electrode_positions[electrode_positions["Labels"]=="F4"]

In [ ]:
plot_cap([1,0,0,1,]+[0,]*16+[1,]+[0,]*33, "Fp1, O1")

In [ ]:
import pandas as pd

ep = pd.read_csv("/mnt/DATA/Giulio_NonLinearityResults/ReRun/EEG/electrode_positions.csv", index_col=0)
ep.x = -ep.x
ep.to_csv("/mnt/DATA/Giulio_NonLinearityResults/ReRun/EEG/electrode_positions.csv", index=False)

In [ ]:
list(average_positions.keys())==chan_lists[0]

In [ ]:
(set(raw.ch_names)-average_positions.keys())-set(bad_electrodes), average_positions.keys()-set(raw.ch_names)

In [ ]:
rs = np.random.default_rng(42)
chan_lists = []
with tqdm(total=tot_subjects, desc="Subjects") as main_pbar:
    for sid in range(first_id, last_id + 1):
        if sid in missing_ids:
            continue

        if False and os.path.isfile(os.path.join(tmp_dir, f"shadow_sub{sid:06}_band5.npy")):
            main_pbar.update()
            continue

        try:
            dirpath = tempfile.mkdtemp()
            with tarfile.open(
                os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"), "r:gz"
            ) as tar:
                tar.extractall(dirpath)
            set_file = os.path.join(dirpath, f"sub-{sid:06}", f"sub-{sid:06}_EC.set")

            raw = mne.io.read_raw_eeglab(set_file).drop_channels(
                bad_electrodes, on_missing="ignore"
            )
            chan_lists.append(raw.ch_names)
            main_pbar.update()
            continue
            boundaries = mne.events_from_annotations(
                raw, regexp="boundary", verbose=False
            )[0][:, 0]

            fs_raw = raw.info["sfreq"]
            for band in bands:
                filtered = []
                power_pieces = []
                shadow_pieces = []
                i_old = 0
                for i in boundaries:
                    if i - i_old < 1250:
                        # discarding segments < 5 s
                        i_old = i
                        continue
                    tmp = raw.copy().crop(i_old / fs_raw, i / fs_raw)
                    tmp.load_data(verbose=False)
                    tmp.filter(
                        l_freq=bands[band][0],
                        h_freq=bands[band][1],
                        method="iir",
                        phase="zero",
                        verbose=False,
                    )

                    tmp_data = tmp.resample(200)._data.T
                    samples = tmp_data.shape[0] // 25

                    power = np.absolute(signal.hilbert(tmp_data, axis=0))
                    dsp = np.average(
                        np.reshape(power[: samples * 25], (-1, 25, power.shape[1])), 1
                    )
                    power_pieces.append(dsp.copy())

                    shadow_data = surrogate(tmp_data, random_state=rs)
                    shadow_power = np.absolute(signal.hilbert(shadow_data, axis=0))
                    dsp = np.average(
                        np.reshape(
                            shadow_power[: samples * 25],
                            (-1, 25, shadow_power.shape[1]),
                        ),
                        1,
                    )
                    shadow_pieces.append(dsp.copy())

                    tmp.resample(fs_band(band))
                    filtered.append(tmp.copy())
                    i_old = i
                filtered[0].append(filtered[1:])
                np.save(
                    os.path.join(tmp_dir, f"filtered_sub{sid:06}_band{band}.npy"),
                    filtered[0]._data.T,
                )
                np.save(
                    os.path.join(tmp_dir, f"power_sub{sid:06}_band{band}.npy"),
                    np.concatenate(power_pieces),
                )
                np.save(
                    os.path.join(tmp_dir, f"shadow_sub{sid:06}_band{band}.npy"),
                    np.concatenate(shadow_pieces),
                )

        finally:
            shutil.rmtree(dirpath)

        main_pbar.update()

In [ ]:
for i in range(1,150):
    if chan_lists[i]!=chan_lists[0]:
        print(i)

In [ ]:
sid = 32301
try:
    dirpath = tempfile.mkdtemp()
    with tarfile.open(
        os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"), "r:gz"
    ) as tar:
        tar.extractall(dirpath)
    set_file = os.path.join(dirpath, f"sub-{sid:06}", f"sub-{sid:06}_EC.set")

    raw = mne.io.read_raw_eeglab(set_file).drop_channels(
        bad_electrodes, on_missing="ignore"
    )
finally:
    shutil.rmtree(dirpath)

In [ ]:
builtin_montages = mne.channels.get_builtin_montages(descriptions=True)
for montage_name, montage_description in builtin_montages:
    print(f"{montage_name}: {montage_description}")

In [ ]:
standard_montage = mne.channels.make_standard_montage("standard_prefixed")
print(standard_montage)

In [ ]:
standard_montage.plot()  # 2D
fig = standard_montage.plot(kind="3d", show=False)  # 3D
fig = fig.gca().view_init(azim=70, elev=15)  # set view angle for tutorial

In [ ]:

for out_name, tmp_name in [
    ("EEG_124s", "filtered"),
    ("electrodeBLP", "power"),
    ("electrodeBLP_shadow", "shadow"),
]:
    for band in bands:
        fs = 8 if "BLP" in out_name else fs_band(band)
        samples_for_chunk = seconds_for_chunk * fs
        data = np.empty([samples_for_chunk * 3, 54, tot_subjects])
        sn = 0
        for sid in range(first_id, last_id + 1):
            if sid in missing_ids:
                continue
            try:
                data[:, :, sn] = np.load(
                    os.path.join(tmp_dir, f"{tmp_name}_sub{sid:06}_band{band}.npy")
                )[: samples_for_chunk * 3, :]
            except:
                print(sid)
            sn += 1
        savemat(
            os.path.join(output_EEG, f"{out_name}_band_{band_names[band]}.mat"),
            {"BLP" if "BLP" in out_name else "EEG": data[:samples_for_chunk, :, :]},
        )
        if "EEG" in out_name:
            for i in range(2, 4):
                savemat(
                    os.path.join(
                        output_EEG, f"{out_name}_band_{band_names[band]}_ses{i}.mat"
                    ),
                    {
                        "EEG": data[
                            (i - 1) * samples_for_chunk : i * samples_for_chunk, :, :
                        ]
                    },
                )

# Searching dataset constraints

In [ ]:
subject_electrodes = {}
mne.set_log_level(verbose="error")
with tqdm(total=tot_subjects, desc="Subjects") as main_pbar:
    for sid in range(first_id, last_id + 1):
        if sid in missing_ids:
            continue

        try:
            dirpath = tempfile.mkdtemp()  # my_tmp_folder(tmp_dir)#
            with tarfile.open(
                os.path.join(output_EEG, f"sub-{sid:06}.tar.gz"), "r:gz"
            ) as tar:
                tar.extractall(dirpath)
            set_file = os.path.join(dirpath, f"sub-{sid:06}", f"sub-{sid:06}_EC.set")

            raw = mne.io.read_raw_eeglab(set_file)
            subject_electrodes[sid] = raw.ch_names
        finally:
            shutil.rmtree(dirpath)

        main_pbar.update()

In [ ]:
raw.set_montage(standard_montage)

In [ ]:
standard_montage.ch_names

In [ ]:
builtin_montages = mne.channels.get_builtin_montages(descriptions=True)
for montage_name, montage_description in builtin_montages:
    elecs = set(mne.channels.make_standard_montage(montage_name).ch_names)
    missing = all_electrodes-elecs
    excess = elecs-all_electrodes
    if len(missing)<10:
        print(montage_name, len(missing), len(excess))
        print(missing)

In [ ]:
all_electrodes = set()
for sid in subject_electrodes:
    all_electrodes.update(subject_electrodes[sid])
all_electrodes

In [ ]:
subject_electrodes_df = pd.DataFrame(
    index=subject_electrodes.keys(), columns=sorted(all_electrodes)
)
candidates = []
for sid in subject_electrodes:
    if set(subject_electrodes[sid]) < all_electrodes:
        candidates.append(sid)
    subject_electrodes_df.loc[sid, subject_electrodes[sid]] = 1

ideal_amount = 61 * 60 * 202

In [ ]:
bad_electrodes = ["T7", "T8", "Cz", "F7", "CP6", "PO10", "Fp2"]
print(
    subject_electrodes_df.loc[
        subject_electrodes_df.T.drop(bad_electrodes).sum(axis=0) < 54
    ].index.to_list()
)

In [ ]:
DELETION_PROB = 0.5
INSERTION_PROB = 0.3


def score(culled):
    tmp = subject_electrodes_df.drop(culled)
    sub_left = tmp.index.size
    chan_left = np.sum(tmp.sum(axis=0) == sub_left)
    return chan_left * (chan_left - 1) * sub_left / ideal_amount


def reproduce(g1, g2):
    l1 = 1 if len(g1) == 1 else np.random.randint(1, len(g1))
    l2 = 1 if len(g2) == 1 else np.random.randint(1, len(g2))
    child = list(
        set(np.random.choice(g1, l1, False)).union(np.random.choice(g2, l2, False))
    )
    if len(child) > 1 and np.random.rand() < DELETION_PROB:
        del child[np.random.randint(len(child))]

    if len(child) < len(candidates) and np.random.rand() < INSERTION_PROB:
        try:
            child.append(np.random.choice(list(set(candidates).difference(child))))
        except:
            print(set(candidates).difference(child), child)

    return child

In [ ]:
DELETION_PROB = 0.5
INSERTION_PROB = 0.5
INSERTION_TRESH = 0.8
DELETION_TRESH = 0.8


def score(culled):
    tmp = subject_electrodes_df.drop(list(culled))
    sub_left = tmp.index.size
    chan_left = np.sum(tmp.sum(axis=0) == sub_left)
    return chan_left * (chan_left - 1) * sub_left / ideal_amount


def reproduce(g1, g2):
    l1 = 1 if len(g1) == 1 else np.random.randint(1, len(g1))
    l2 = 1 if len(g2) == 1 else np.random.randint(1, len(g2))
    child = list(
        set(np.random.choice(g1, l1, False)).union(np.random.choice(g2, l2, False))
    )

    max_insert = len(candidates) - len(child)
    if max_insert > 1 and np.random.rand() < INSERTION_PROB:
        thresh = INSERTION_TRESH
        for i in range(max_insert):
            if np.random.rand() > thresh:
                break
            try:
                child.append(np.random.choice(list(set(candidates).difference(child))))
            except:
                print(set(candidates).difference(child), child)
            thresh *= INSERTION_TRESH
    max_delete = len(child) - 1
    if max_delete and np.random.rand() < DELETION_PROB:
        thresh = DELETION_TRESH
        for i in range(max_delete):
            if np.random.rand() > thresh:
                break
            try:
                del child[np.random.randint(len(child))]
            except:
                print(child)
            thresh *= DELETION_TRESH

    return tuple(child)

In [ ]:
pop_sizes = np.random.randint(1, len(candidates), 10000)
population = []
for s in pop_sizes:
    population.append(tuple(np.random.choice(candidates, s, False)))
best = [0, [], -1]

In [ ]:
with multiprocessing.Pool(45) as polla:
    for gen in tqdm(range(0, 50000), desc="Generation", initial=0, total=50000):
        scores = np.array(polla.map(score, population))
        if max(scores) > best[0]:
            best = [max(scores), population[np.argmax(scores)], gen]
            with open("genetic_straight.txt", "a") as file:
                print("New best: ", best, "\n", file=file)

        if not gen % 20:
            print(
                f"Gen: {gen} Best{[round(max(scores),5), len(population[np.argmax(scores)])]}, Avg genes: {np.mean([len(c) for c in population])}, {np.sum(np.argsort(scores)[-500:]>499)}\n\tAll times best:{[round(best[0],5),len(best[1]), best[2]]}"
            )
            with open("genetic_straight.txt", "a") as file:
                print(
                    f"Gen: {gen} Best{[round(max(scores),5), len(population[np.argmax(scores)])]}, Avg genes: {np.mean([len(c) for c in population])}, {np.sum(np.argsort(scores)[-500:]>499)}\n\tAll times best:{[round(best[0],5),len(best[1]), best[2]]}",
                    file=file,
                )

        scores /= np.sum(scores)
        new_pop = [population[i] for i in np.argsort(scores)[-500:]]
        genitori = [
            np.random.choice(len(population), 2, False, scores) for i in range(9500)
        ]
        for child in polla.starmap(
            reproduce, ((population[g1], population[g2]) for g1, g2 in genitori)
        ):
            new_pop.append(child)
        population = new_pop
        if not gen % 20:
            print("Cleaning")
            population = list(set(population))
            if len(population) < 10000:
                to_get = 10000 - len(population)
                print(f"Removed {to_get} duplicates.")
                genitori = [
                    np.random.choice(len(population), 2, False) for i in range(to_get)
                ]
                for child in polla.starmap(
                    reproduce, ((population[g1], population[g2]) for g1, g2 in genitori)
                ):
                    population.append(child)

In [ ]:
elec_sums = subject_electrodes_df.sum(axis=0)
for elec in elec_sums.index:
    if elec_sums.loc[elec] == subject_electrodes_df.index.size:
        subject_electrodes_df.drop(columns=elec, inplace=True)

In [ ]:
import matplotlib.pyplot as plt

mappa = np.nan_to_num(subject_electrodes_df.values.astype(float))
as1 = np.argsort(np.sum(mappa, 0))
as0 = np.argsort(np.sum(mappa, 1))
plt.imshow(mappa[as0, :][:, as1])

In [ ]:
shapes = {i: [] for i in range(1, 6)}
with tqdm(total=tot_subjects, desc="Subjects") as main_pbar:
    for sid in range(first_id, last_id + 1):
        if sid in missing_ids:
            continue
        for band in range(1, 6):
            tmp = np.load(os.path.join(tmp_dir, f"filtered_sub{sid:06}_band{band}.npy"))
            shapes[band].append(tmp.shape)
        main_pbar.update()

In [ ]:
shapes = {i: np.array(shapes[i]) for i in shapes}
for i in shapes:
    print(f"Band {band_names[i]}\n\tLengths")
    lengths, hm = np.unique(shapes[i][:, 0], False, False, True, 0)
    lower = np.sum(hm[np.where(lengths < lengths[np.argmax(hm)])])
    print(
        f"Most common: {lengths[np.argmax(hm)]} [{(lengths[np.argmax(hm)]/fs_band(i))//3}] ({np.max(hm)}/{tot_subjects-lower})"
    )
    print(
        f"Shortest: {np.min(lengths)} [{(np.min(lengths)/fs_band(i))//3}] ({hm[np.argmin(lengths)]})"
    )
    print(f"Total below mode: {lower}")
    print("\tElectrodes")
    electrodes, hm = np.unique(shapes[i][:, 1], False, False, True, 0)
    print(f"Most common: {electrodes[np.argmax(hm)]} ({np.max(hm)})")
    print(f"Shortest: {np.min(electrodes)} ({hm[np.argmin(electrodes)]})")
    print(
        f"Total below mode: {np.sum(hm[np.where(electrodes<electrodes[np.argmax(hm)])])}"
    )

In [ ]:
np.unique(shapes[1][:, 0], False, False, True, 0)

In [ ]:
3367 / fs_band(1)

In [ ]:


def my_tmp_folder(dir):
    name = "tmp" + base64.b64encode(np.random.rand(1).tobytes()).decode()[:-2].replace(
        "/", "_"
    )
    mytmpdir = os.path.join(dir, name)
    os.mkdir(mytmpdir)
    return mytmpdir